# NEID L2: querying and inspecting RV products

This is the most common NEID workflow: search the **Level-2** holdings over a
date range, keep just the columns you care about, and look at what comes back.
By the end you will have an {term}`astropy Table` of radial velocities ready to
plot or fit.

```{note}
This notebook runs against the live NEID server. The published version here
uses synthetic data so it builds without an archive account — the numbers shown
are illustrative, not real observations.
```

## What is NEID L2?

{term}`NEID` is a high-precision {term}`radial velocity (RV)` spectrograph. Its
archive offers files at several {term}`data level`s — a ladder of how far the
data has been processed:

- {term}`L0` — raw detector frames, straight off the CCD.
- {term}`L1` — calibrated 1D {term}`extracted spectrum` (flux vs. wavelength).
- {term}`L2` — RV-ready products: the {term}`CCF (cross-correlation function)`
  and the radial velocity derived from it.

Most science users want **L2**, because that is where the radial velocities
live. The pipeline measures each RV by cross-correlating the star's spectrum
against a template of known stellar lines and finding the Doppler shift of the
result; you never compute any of that yourself. For the full story see
{doc}`../concepts/data-levels` and {doc}`../concepts/radial-velocities`.

In [ ]:
import os

from astropy.table import Table

OFFLINE = bool(os.getenv("NEXSCI_RV_OFFLINE_FIXTURES"))

if OFFLINE:
    from unittest.mock import patch

    from astropy.coordinates import SkyCoord

    _fake = Table(
        {
            "l2filename": [
                "neid20241101_0001.fits",
                "neid20241101_0002.fits",
                "neid20241102_0001.fits",
            ],
            "date_obs": ["2024-11-01T03:12:00", "2024-11-01T03:25:00", "2024-11-02T03:18:00"],
            "object": ["HD 189733", "HD 189733", "HD 189733"],
            "exptime": [300.0, 300.0, 300.0],
            "ccfrvmod": [-2.276, -2.279, -2.281],
        }
    )
    _patcher = patch("nexsci_rv.tap.TapClient.query", return_value=_fake)
    _patcher.start()
    # Stub Simbad/NED name resolution so query_criteria(object=...) works offline.
    _fake_coord = SkyCoord("20h00m43.7s", "+22d42m39s")
    _resolver_patcher = patch("nexsci_rv.base.SkyCoord.from_name", return_value=_fake_coord)
    _resolver_patcher.start()
    print("Offline mode: queries return synthetic data.")

## Build the archive client

`NEIDArchive()` with no arguments points at the IPAC production server. Public
L2 data is queryable without `login()`.


In [ ]:
from nexsci_rv import NEIDArchive

neid = NEIDArchive()
neid

## Query by date range

All query methods return an `astropy.table.Table`. Use `columns=` to keep only
what you need; the package validates column names against the table's static
schema before any HTTP request.


In [ ]:
tbl = neid.query_datetime(
    "l2",
    ("2024-11-01", "2024-11-06"),
    columns=["l2filename", "date_obs", "object", "exptime", "ccfrvmod"],
    maxrec=20,
)
tbl

Each row is one observation. The columns we asked for:

- `l2filename` — the name of the L2 {term}`FITS` file; you need it (with
  `l2filepath`) to {term}`download` the data later.
- {term}`date_obs / obsdate` — the observation time stamp (`date_obs` on NEID);
  this is the column {term}`query_datetime` filters on.
- {term}`object` — the free-text target name the observer typed in.
- {term}`exptime` — exposure time in seconds.
- {term}`ccfrvmod` — **the answer**: the radial velocity in m/s, the number you
  would phase-fold and fit for a planet (its error bar is {term}`ccfnoise`).

For the full column tour see {doc}`../concepts/results-columns`; `describe_table`
(below) lists every column the table offers.

```{tip}
We searched by **date range** here because a time window is the safest first
query — it always works. When you know a star by name, RA/Dec, a catalog ID, a
program, or a PI instead, there is a dedicated `query_*` method for each. See
{doc}`../concepts/choosing-a-query` for picking the right one.

`maxrec` caps how many rows the server returns (here, 20). Keep it small while
you are still working out a query, then raise it once the results look right.
```

Most NEID data is public and needs no login. Reaching data still inside its
{term}`proprietary period` requires `neid.login()` with IPAC NEID credentials —
see {doc}`../concepts/downloading-and-proprietary`.

## Multi-criterion query in one round-trip

`query_criteria` composes constraints into a single ADQL query — one HTTP
request even with several filters.

In [ ]:
from astropy import units as u

tbl2 = neid.query_criteria(
    {
        "data_level": "l2",
        "datetime": ("2024-11-01", "2024-11-06"),
        "object": "HD 189733",
        "object_radius": 30 * u.arcsec,
    },
    columns=["l2filename", "date_obs", "object", "ccfrvmod"],
    maxrec=20,
)
tbl2

## Inspecting the schema

`describe_table` returns the server-side `TAP_SCHEMA` metadata for a given
table — useful for finding columns that aren't in the package's static map.

In [ ]:
if not OFFLINE:
    schema = neid.describe_table("neidl2")
    schema[:10]
else:
    print("(skipped in offline mode)")

## What next

You now have a table of NEID radial velocities. From here:

- {doc}`02_expres_solar` — the same query pattern applied to the EXPRES solar
  RV time series.
- {doc}`03_download_and_plot` — turn a result table into FITS files on disk and
  open a spectrum.
- {doc}`../quickstart` — every query pattern at a glance.